# Lab4: Recommendation System and BERT
## 1. Recommendation system overview
A recommendation system predicts what items a user may like and returns a ranked list of items. Examples:
| Platform           | User              | Item                  | Example feedback signals                             | Recommendation goal                      |
| ------------------ | ----------------- | --------------------- | ---------------------------------------------------------------------------- | ---------------------------------------- |
| Netflix            | viewer            | movie/show            | watch history, watch time, ratings, likes/dislikes, skips, searches          | what to watch next                       |
| Amazon             | customer          | product               | clicks, purchases, add-to-cart, wish lists, reviews, ratings, search queries | what to buy next                         |
| Spotify            | listener          | song/playlist/podcast | listening history, skips, replays, likes, playlist saves, follows            | what to listen to next                   |
| YouTube            | viewer            | video                 | clicks, watch time, likes/dislikes, comments, subscriptions, search history  | what to watch next                       |
| TikTok / Instagram | viewer            | short video/post      | views, watch time, replays, likes, shares, comments, follows, skips          | what content to show next                |
| LinkedIn           | professional user | job/post/connection   | clicks, applications, profile views, follows, reactions, connection requests | what jobs, posts, or people to recommend |


Recommendation systems may utilize both explicit feedback and implicit feedback as signals:
| Explicit feedback | Implicit feedback |
| ----------------- | ----------------- |
| Star rating       | Click             |
| Review score      | Watch time        |
| Like/dislike      | Add to cart       |
| Survey response   | Purchase          |

There are 3 common recommendation approaches
1. Popularity baseline
2. Collaborative filltering
3. Content-based recommendation


## 2. Recommendation on MovieLens 1M data
### 2.1. AWS S3
Amazon S3 (Simple Storage Service) is AWS’s object storage service.
- Object = data + metadata

Here are some key terms:

- **Bucket**: the top-level container (globally unique name, for example, `de300-tutorial-data`).

- Object: one stored file (e.g., `movies.dat`).

- Key: the “path-like name” of the object inside a bucket (e.g., `s3://de300-tutorial-data/datasets/movielens/ml-1m.zip`).

- Prefix: the leading part of a key used for organization. S3 doesn’t have real folders; the console shows folder-like views using prefixes.

We may upload/download/modify files on s3 through AWS CLI tool (which may be installed through instruction on [AWS CLI installation](https://docs.aws.amazon.com/cli/latest/userguide/getting-started-install.html)).

Before we use AWS CLI to get access to data on S3, we should set up credentials on the device (your own laptop or EC2). One of the ways to do so is go to the [https://nu-sso.awsapps.com/](https://nu-sso.awsapps.com/), click `Access Keys` and copy AWS Environmental Variables, e.g.,
```
```

Some commands that are frequently used:
1. List all objects under a prefix
   ```
   aws s3 ls s3://dinglin-spring26/
   ```
2. Copy ONE object from S3 to local
   ```
   aws s3 cp s3://dinglin-spring26/lab4/ml-1m/movies.dat ./movies.dat
   ```

3. Copy a “folder” (prefix) recursively
   ```
   aws s3 cp s3://dinglin-spring26/lab4/ml-1m/ ./lab4-data/ --recursive
   ```

4. Copy files and folders among different buckets
   ```
   aws s3 cp s3://dinglin-spring26/lab4/ml-1m/movies.dat s3://DEST_BUCKET/path/to/movies.dat
   aws s3 cp s3://dinglin-spring26/lab4/ml-1m/ s3://DEST_BUCKET/some/prefix/ --recursive
   ```

### 2.2. Dataset Intro
MovieLens 1M is a classic benchmark dataset for recommender systems released by the GroupLens Research Project. It contains 1,000,209 ratings (1–5 stars, whole-star only) from 6,040 users on movie IDs up to 3,952, and each user has at least 20 ratings.

### 2.3. Data Loading
We will read `ratings.dat` and `movies.dat` from the `ml-1m/` folder. These files use `::` as a delimiter, so we set `engine='python'` in `pandas.read_csv`.

- ratings.dat: (user_id, movie_id, rating, timestamp)
- movies.dat: (movie_id, title, genres)

In [1]:
import os
os.chdir(r"..")
os.getcwd()

'/Users/evanbarnett/Desktop/Northwestern/Classes/de300/data_eng300SP_barnett/lab_notebooks'

In [2]:
if os.path.exists("data/ml-1m/ratings.dat") and os.path.exists("data/ml-1m/movies.dat"):
    print("MovieLens 1M data already available in data/ml-1m/.")
else:
    from datarec.datasets import Movielens
    Movielens(version="1m", folder="data").prepare()

MovieLens 1M data already available in data/ml-1m/.


In [3]:
import pandas as pd

DATA_DIR = "data/ml-1m/"
RATINGS_FILE = DATA_DIR + "ratings.dat"
MOVIES_FILE = DATA_DIR + "movies.dat"

ratings = pd.read_csv(RATINGS_FILE, sep="::", engine="python",
                      names=["user_id","movie_id","rating","timestamp"])
movies  = pd.read_csv(MOVIES_FILE,  sep="::", engine="python",
                      names=["movie_id","title","genres"], encoding="latin-1")

ratings.head(), movies.head()

(   user_id  movie_id  rating  timestamp
 0        1      1193       5  978300760
 1        1       661       3  978302109
 2        1       914       3  978301968
 3        1      3408       4  978300275
 4        1      2355       5  978824291,
    movie_id                               title                        genres
 0         1                    Toy Story (1995)   Animation|Children's|Comedy
 1         2                      Jumanji (1995)  Adventure|Children's|Fantasy
 2         3             Grumpier Old Men (1995)                Comedy|Romance
 3         4            Waiting to Exhale (1995)                  Comedy|Drama
 4         5  Father of the Bride Part II (1995)                        Comedy)

Implicit recommendation models usually need positive interactions rather than 1–5 star ratings. Here we treat ratings ≥ 4 as a positive signal and create a binary column value=1.0

In [4]:
pos = ratings[ratings["rating"] >= 4].copy()
pos["value"] = 1.0

To evaluate recommendation, we hold out each user’s most recent positive interaction as the test item (using timestamp). The remaining positives become training data.

In [5]:
pos = pos.sort_values(["user_id","timestamp"])
test = pos.groupby("user_id").tail(1)
train = pos.drop(test.index)

train.shape, test.shape


((569243, 5), (6038, 5))

Before building sophisticated models, it’s useful to compute a simple baseline: recommend the most frequently liked movies in the training set. This gives you a sanity check for your pipeline.

In [6]:
top_movies = train.groupby("movie_id")["value"].sum().sort_values(ascending=False)
top10 = top_movies.head(10).index.tolist()
movies[movies["movie_id"].isin(top10)][["movie_id","title","genres"]]


,movie_id,title,genres
257,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Fantasy|Sci-Fi
589,593,"Silence of the Lambs, The (1991)",Drama|Thriller
604,608,Fargo (1996),Crime|Drama|Thriller
1178,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War
1180,1198,Raiders of the Lost Ark (1981),Action|Adventure
1192,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Romance|Sci-Fi|War
1959,2028,Saving Private Ryan (1998),Action|Drama|War
2502,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller
2693,2762,"Sixth Sense, The (1999)",Thriller
2789,2858,American Beauty (1999),Comedy|Drama


### 2.4. ALS (Collborative Filltering)
Many recommenders (including `implicit ALS`) expect a sparse matrix. We map raw IDs to contiguous indices:

- user2idx: user_id → row index
- item2idx: movie_id → column index
Then we create a sparse matrix X_ui with shape [num_users, num_items].

In [7]:
import numpy as np
from scipy.sparse import coo_matrix

user_ids = train["user_id"].unique()
movie_ids = train["movie_id"].unique()

user2idx = {u:i for i,u in enumerate(user_ids)}
movie_ids = np.array(movie_ids)  # ensure it's a numpy array
item2idx = {int(m): i for i, m in enumerate(movie_ids)}
idx2item = {i: int(m) for i, m in enumerate(movie_ids)}

rows = train["user_id"].map(user2idx).to_numpy()
cols = train["movie_id"].map(item2idx).to_numpy()
data = train["value"].to_numpy().astype(np.float32)
X_ui = coo_matrix((data, (rows, cols)), shape=(len(user_ids), len(movie_ids))).tocsr()


In [8]:
!pip install implicit

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.org/simple/



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [9]:
import implicit

model = implicit.als.AlternatingLeastSquares(
    factors=64, regularization=0.01, iterations=20
)

# implicit expects item-user matrix
model.fit(X_ui)


/Users/evanbarnett/Desktop/Northwestern/Classes/de300/data_eng300SP_barnett/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



  0%|          | 0/20 [00:00<?, ?it/s]


 10%|█         | 2/20 [00:00<00:01, 11.81it/s]


 20%|██        | 4/20 [00:00<00:01, 11.86it/s]


 30%|███       | 6/20 [00:00<00:01, 11.11it/s]

...


 65%|██████▌   | 13/20 [00:01<00:00,  9.52it/s]


 75%|███████▌  | 15/20 [00:01<00:00, 10.16it/s]


 85%|████████▌ | 17/20 [00:01<00:00, 10.67it/s]


 95%|█████████▌| 19/20 [00:01<00:00,  9.97it/s]


100%|██████████| 20/20 [00:01<00:00, 10.11it/s]

In [10]:
u = int(user_ids[0])
u_idx = int(user2idx[u])

# IMPORTANT: pass X_ui[u_idx] (1 row), not the full X_ui
recs = model.recommend(u_idx, X_ui[u_idx], N=10)

# Handle implicit returning either (ids, scores) OR list of pairs
if isinstance(recs, tuple) and len(recs) == 2:
    item_ids, scores = recs
else:
    item_ids = [i for i, s in recs]
    scores  = [s for i, s in recs]

rec_movie_ids = [idx2item[int(i)] for i in item_ids]

rec_df = movies.set_index("movie_id").loc[rec_movie_ids][["title", "genres"]].reset_index()
rec_df["score"] = list(scores)
rec_df


,movie_id,title,genres,score
0,364,"Lion King, The (1994)",Animation|Children's|Musical,0.669994
1,318,"Shawshank Redemption, The (1994)",Drama,0.496988
2,2081,"Little Mermaid, The (1989)",Animation|Children's|Comedy|Musical|Romance,0.452305
3,1225,Amadeus (1984),Drama,0.409505
4,34,Babe (1995),Children's|Comedy|Drama,0.385542
5,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War,0.350312
6,1968,"Breakfast Club, The (1985)",Comedy|Drama,0.343699
7,953,It's a Wonderful Life (1946),Drama,0.341226
8,2085,101 Dalmatians (1961),Animation|Children's,0.339613
9,2087,Peter Pan (1953),Animation|Children's|Fantasy|Musical,0.339604


### 2.5. Evaluation: Recall@K and NDCG@K
We evaluate each user by checking whether the held-out test movie appears in the top-K recommendations:

- Recall@K: fraction of users whose held-out item is retrieved in top-K.
- NDCG@K: like Recall@K, but gives higher credit when the item appears closer to rank 1.

In [11]:
import math

def ndcg_at_k(rank, k):
    if rank is None or rank >= k:
        return 0.0
    return 1.0 / math.log2(rank + 2)

def _get_rec_items(recs):
    # implicit may return (item_ids, scores) OR list of (id, score)
    if isinstance(recs, tuple) and len(recs) == 2:
        item_ids = recs[0]
        return [int(i) for i in item_ids]
    else:
        return [int(i) for i, _ in recs]

def eval_model(model, X_ui, test_df, K=10):
    hits, ndcgs, n = 0, 0.0, 0
    for u, gt in zip(test_df["user_id"], test_df["movie_id"]):
        if (u not in user2idx) or (gt not in item2idx):
            continue

        u_idx = int(user2idx[u])

        # IMPORTANT: pass only this user's row
        recs = model.recommend(u_idx, X_ui[u_idx], N=K)

        rec_items = _get_rec_items(recs)
        gt_idx = int(item2idx[gt])

        if gt_idx in rec_items:
            hits += 1
            ndcgs += ndcg_at_k(rec_items.index(gt_idx), K)

        n += 1

    return {"Recall@K": hits / max(n, 1), "NDCG@K": ndcgs / max(n, 1), "Users": n}

eval_model(model, X_ui, test, K=10)

{'Recall@K': 0.09297315213788532, 'NDCG@K': 0.0462123414827576, 'Users': 6034}

### 2.6. Content Embedding with BERT Model

In [12]:
movies["text"] = movies["title"].fillna("") + " [SEP] " + movies["genres"].fillna("")
movies["text"].head()


0    Toy Story (1995) [SEP] Animation|Children's|Co...
1    Jumanji (1995) [SEP] Adventure|Children's|Fantasy
2         Grumpier Old Men (1995) [SEP] Comedy|Romance
3          Waiting to Exhale (1995) [SEP] Comedy|Drama
4      Father of the Bride Part II (1995) [SEP] Comedy
Name: text, dtype: object

In [13]:
import torch
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
BERT_MODEL_NAME = "bert-base-uncased"
BERT_EMB_PATH = "data/ml-1m/item_emb_with_ids.pt"

if os.path.exists(BERT_EMB_PATH):
    obj = torch.load(BERT_EMB_PATH, map_location="cpu", weights_only=False)
    item_emb = obj["item_emb"]
else:
    tok = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
    bert = AutoModel.from_pretrained(BERT_MODEL_NAME).to(device)
    bert.eval()

    @torch.no_grad()
    def encode_texts(texts, batch_size=64, max_len=64):
        embs = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            inp = tok(batch, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(device)
            out = bert(**inp).last_hidden_state[:,0,:]  # [CLS]
            embs.append(out.cpu())
        return torch.cat(embs, dim=0)

    item_emb = encode_texts(movies["text"].tolist())

item_emb.shape


torch.Size([3883, 768])

### 2.6. Save item embedding as tensor

In [14]:
os.getcwd()

'/Users/evanbarnett/Desktop/Northwestern/Classes/de300/data_eng300SP_barnett/lab_notebooks'

In [15]:
save_path = "data/ml-1m/item_emb.pt"
torch.save(item_emb, save_path)

# later
item_emb2 = torch.load(save_path, map_location="cpu")
print(item_emb2.shape)


torch.Size([3883, 768])


In [16]:
payload = {
    "item_emb": item_emb,                      # [num_items, hidden_dim]
    "movie_id": movies["movie_id"].to_numpy(), # same order as embeddings
}
torch.save(payload, "data/ml-1m/item_emb_with_ids.pt")

# later
obj = torch.load("data/ml-1m/item_emb_with_ids.pt", map_location="cpu", weights_only=False)
item_emb = obj["item_emb"]
movie_id = obj["movie_id"]

In [17]:
import torch.nn.functional as F

# normalize for cosine similarity
E = F.normalize(item_emb, p=2, dim=1)

movieid_to_row = {mid:i for i,mid in enumerate(movies["movie_id"].tolist())}

def similar_movies(movie_id, topk=10):
    i = movieid_to_row[movie_id]
    sims = (E @ E[i]).numpy()
    top = sims.argsort()[::-1][1:topk+1]  # exclude itself
    return movies.iloc[top][["movie_id","title","genres"]]

similar_movies(movie_id=1, topk=10)  # Toy Story is movie_id=1 in ml-1m


,movie_id,title,genres
3045,3114,Toy Story 2 (1999),Animation|Children's|Comedy
3682,3751,Chicken Run (2000),Animation|Children's|Comedy
1050,1064,Aladdin and the King of Thieves (1996),Animation|Children's|Comedy
584,588,Aladdin (1992),Animation|Children's|Comedy|Musical
2286,2355,"Bug's Life, A (1998)",Animation|Children's|Comedy
1743,1806,Paulie (1998),Adventure|Children's|Comedy
2069,2138,Watership Down (1978),Animation|Children's|Drama|Fantasy
591,595,Beauty and the Beast (1991),Animation|Children's|Musical
2072,2141,"American Tail, An (1986)",Animation|Children's|Comedy
451,455,Free Willy (1993),Adventure|Children's|Drama


In [18]:
from collections import defaultdict
import numpy as np

liked = train.groupby("user_id")["movie_id"].apply(list).to_dict()

def recommend_for_user_content(user_id, topk=10):
    mids = liked.get(user_id, [])
    mids = [m for m in mids if m in movieid_to_row]
    if not mids: 
        return None
    rows = [movieid_to_row[m] for m in mids]
    u = E[rows].mean(dim=0, keepdim=True)
    sims = (E @ u[0]).numpy()
    # filter already seen
    seen_rows = set(rows)
    candidates = [i for i in sims.argsort()[::-1] if i not in seen_rows]
    top = candidates[:topk]
    return movies.iloc[top][["movie_id","title","genres"]]

recommend_for_user_content(user_ids[0], topk=10)


,movie_id,title,genres
3038,3107,Backdraft (1991),Action|Drama
3099,3168,Easy Rider (1969),Adventure|Drama
1096,1112,Palookaville (1996),Action|Drama
651,657,Yankee Zulu (1994),Comedy|Drama
1885,1954,Rocky (1976),Action|Drama
2857,2926,Hairspray (1988),Comedy|Drama
2508,2577,Metroland (1997),Comedy|Drama
3096,3165,Boiling Point (1993),Action|Drama
533,537,Sirens (1994),Comedy|Drama
3457,3526,Parenthood (1989),Comedy|Drama


In [19]:
# item2idx maps movie_id -> item_idx used by X_ui / eval_model
idx2item = {i: m for m, i in item2idx.items()}
movieid_to_row = {mid: i for i, mid in enumerate(movies["movie_id"].tolist())}

def build_item_embedding_matrix(item_emb, item2idx, movieid_to_row):
    """Return embeddings in the same item-index order used by X_ui."""
    E_all = F.normalize(item_emb.float(), p=2, dim=1)
    idx2item_local = {i: m for m, i in item2idx.items()}
    num_items = len(item2idx)
    dim = E_all.shape[1]
    E_items = torch.zeros((num_items, dim), dtype=E_all.dtype)

    missing = 0
    for item_idx in range(num_items):
        mid = idx2item_local[item_idx]
        row = movieid_to_row.get(mid)
        if row is None:
            missing += 1
        else:
            E_items[item_idx] = E_all[row]

    return E_items, missing

E_items, missing = build_item_embedding_matrix(item_emb, item2idx, movieid_to_row)
print("missing movies:", missing, "out of", len(item2idx))

missing movies: 0 out of 3530


### 2.7. Compare Two Pre-trained Text Encoders
The same content-recommendation logic can use any text encoder as long as we convert every movie into an embedding and reorder those embeddings to match the sparse matrix item indices.

## Lab Assignment
1. Complete the recommend functions powered by BERT Content Embedding.
2. Try at least 2 different pre-trained model for embedding and test the model performance.

In [23]:
SECOND_MODEL_NAME = "distilbert-base-uncased"
DISTILBERT_EMB_PATH = "data/ml-1m/distilbert_item_emb_with_ids.pt"

def load_tokenizer_and_model(model_name):
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)
        encoder = AutoModel.from_pretrained(model_name, local_files_only=True)
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        encoder = AutoModel.from_pretrained(model_name)
    return tokenizer, encoder.to(device).eval()

@torch.no_grad()
def encode_texts_with_model(model_name, texts, batch_size=64, max_len=64):
    tokenizer, encoder = load_tokenizer_and_model(model_name)
    embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inp = tokenizer(batch, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(device)
        token_emb = encoder(**inp).last_hidden_state
        mask = inp["attention_mask"].unsqueeze(-1)
        pooled = (token_emb * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        embs.append(pooled.cpu())

    del encoder
    if device == "cuda":
        torch.cuda.empty_cache()
    return torch.cat(embs, dim=0)


In [20]:
class BertContentModel:
    def __init__(self, E_items):
        # E_items is ordered by the item indices used in X_ui.
        self.E = F.normalize(E_items.float(), p=2, dim=1)

    def recommend(self, u_idx, X_ui, N=10):
        # eval_model passes a single sparse user row; this also handles a full matrix.
        user_row = X_ui.getrow(u_idx) if hasattr(X_ui, "getrow") and X_ui.shape[0] > 1 else X_ui
        seen_items = np.asarray(user_row.indices, dtype=np.int64)
        if len(seen_items) == 0:
            return []

        user_emb = self.E[seen_items].mean(dim=0, keepdim=True)
        user_emb = F.normalize(user_emb, p=2, dim=1).squeeze(0)

        scores = (self.E @ user_emb).clone()
        scores[torch.as_tensor(seen_items, dtype=torch.long)] = -torch.inf

        k = min(N, scores.numel() - len(seen_items))
        if k <= 0:
            return []

        top_scores, top_idx = torch.topk(scores, k=k)
        return [(int(i), float(s)) for i, s in zip(top_idx.tolist(), top_scores.tolist())]

In [21]:
bert_model = BertContentModel(E_items)

In [24]:
if os.path.exists(DISTILBERT_EMB_PATH):
    distilbert_obj = torch.load(DISTILBERT_EMB_PATH, map_location="cpu", weights_only=False)
    distilbert_emb = distilbert_obj["item_emb"]
else:
    distilbert_emb = encode_texts_with_model(SECOND_MODEL_NAME, movies["text"].tolist())
    torch.save(
        {"item_emb": distilbert_emb, "movie_id": movies["movie_id"].to_numpy()},
        DISTILBERT_EMB_PATH,
    )

distilbert_emb.shape

torch.Size([3883, 768])

In [25]:
distilbert_E_items, distilbert_missing = build_item_embedding_matrix(distilbert_emb, item2idx, movieid_to_row)
print("missing movies:", distilbert_missing, "out of", len(item2idx))

distilbert_model = BertContentModel(distilbert_E_items)
distilbert_metrics = eval_model(distilbert_model, X_ui, test, K=10)
distilbert_metrics

missing movies: 0 out of 3530


{'Recall@K': 0.013755386145177328,
 'NDCG@K': 0.006293593514770869,
 'Users': 6034}

In [22]:
bert_metrics = eval_model(bert_model, X_ui, test, K=10)
bert_metrics

{'Recall@K': 0.01557838912827312,
 'NDCG@K': 0.007848329700857354,
 'Users': 6034}

In [26]:
content_results = pd.DataFrame([
    {"embedding_model": "bert-base-uncased", **bert_metrics},
    {"embedding_model": SECOND_MODEL_NAME, **distilbert_metrics},
])
content_results

,embedding_model,Recall@K,NDCG@K,Users
0,bert-base-uncased,0.015578,0.007848,6034
1,distilbert-base-uncased,0.013755,0.006294,6034
